In [1]:
import pandas as pd
ruta = "../data/raw/INM_estatal_dic25.csv"
df = pd.read_csv(ruta, dtype={"clave_ent": str})
df.head()

,anio,clave_ent,entidad,bien_juridico_afectado,tipo_delito,subtipo_delito,modalidad,mes,fecha,incidencia_delictiva,entidad_federativa
0,2015,01,Aguascalientes,El patrimonio,Abuso de confianza,Abuso de confianza,Abuso de confianza,Abril,2015-04-01,22,Aguascalientes
1,2015,01,Aguascalientes,El patrimonio,Abuso de confianza,Abuso de confianza,Abuso de confianza,Agosto,2015-08-01,40,Aguascalientes
2,2015,01,Aguascalientes,El patrimonio,Abuso de confianza,Abuso de confianza,Abuso de confianza,Diciembre,2015-12-01,26,Aguascalientes
3,2015,01,Aguascalientes,El patrimonio,Abuso de confianza,Abuso de confianza,Abuso de confianza,Enero,2015-01-01,41,Aguascalientes
4,2015,01,Aguascalientes,El patrimonio,Abuso de confianza,Abuso de confianza,Abuso de confianza,Febrero,2015-02-01,33,Aguascalientes


In [2]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/processed/fact_incidencia_mensual.csv", dtype={"clave_entidad": str})

In [3]:
comp = df.pivot_table(index=["clave_entidad", "estado"], columns="anio",
                      values="incidencia", aggfunc="sum", fill_value=0).reset_index()
comp.columns.name = None
comp = comp.rename(columns={2024: "total_2024", 2025: "total_2025"})
comp.head()

,clave_entidad,estado,total_2024,total_2025
0,01,Aguascalientes,42989,39291
1,02,Baja California,107397,91936
2,03,Baja California Sur,25000,24740
3,04,Campeche,12434,5580
4,05,Coahuila,61636,69149


In [4]:
comp["variacion_abs"] = comp["total_2025"] - comp["total_2024"]
comp["variacion_pct"] = comp["variacion_abs"] / comp["total_2024"]

comp["clasificacion"] = np.select(
    [comp["variacion_abs"] < 0, comp["variacion_abs"] > 0],
    ["Positivo (bajó)", "Negativo (subió)"],
    default="Sin cambio")

comp = comp.sort_values("variacion_abs")
comp["clasificacion"].value_counts()

clasificacion
Positivo (bajó)     22
Negativo (subió)    10
Name: count, dtype: int64

In [5]:
comp.to_csv("../data/processed/comparativo_por_estado.csv", index=False, encoding="utf-8-sig")

In [6]:
def comparar(df, por):
    comp = df.pivot_table(index=por, columns="anio", values="incidencia",
                          aggfunc="sum", fill_value=0).reset_index()
    comp.columns.name = None
    comp = comp.rename(columns={2024: "total_2024", 2025: "total_2025"})

    comp["variacion_abs"] = comp["total_2025"] - comp["total_2024"]
    comp["variacion_pct"] = comp["variacion_abs"] / comp["total_2024"]
    comp["variacion_pct"] = comp["variacion_pct"].replace([np.inf, -np.inf], np.nan)

    comp["clasificacion"] = np.select(
        [comp["variacion_abs"] < 0, comp["variacion_abs"] > 0],
        ["Positivo (bajó)", "Negativo (subió)"],
        default="Sin cambio")

    return comp.sort_values("variacion_abs")

In [7]:
por_delito = comparar(df, ["bien_juridico", "tipo_delito"])
por_delito.to_csv("../data/processed/comparativo_por_delito.csv", index=False, encoding="utf-8-sig")
por_delito.shape

(40, 7)

In [8]:
detalle = comparar(df, ["clave_entidad", "estado", "bien_juridico", "tipo_delito"])
detalle.to_csv("../data/processed/comparativo_estado_delito.csv", index=False, encoding="utf-8-sig")
detalle.shape

(1280, 9)

In [9]:
detalle["base_suficiente"] = detalle["total_2024"] >= 30

positivos = detalle[detalle["clasificacion"] == "Positivo (bajó)"]
negativos = detalle[detalle["clasificacion"] == "Negativo (subió)"]

positivos.to_csv("../data/processed/positivos_bajaron.csv", index=False, encoding="utf-8-sig")
negativos.to_csv("../data/processed/negativos_subieron.csv", index=False, encoding="utf-8-sig")
len(positivos), len(negativos)

(601, 507)